# QLoRA Fine-Tuning: LLaMA-2-7B-Chat for HR Policy Assistant

**Domain:** Sri Lankan HR Policies & Labor Law  
**Model:** `meta-llama/Llama-2-7b-chat-hf`  
**Method:** QLoRA (4-bit NF4 Quantization + LoRA Adapters)  
**Author:** Chamindu Nipun  
**Date:** 27/02/2026

---

## Notebook Overview

| Step | Description |
|------|-------------|
| 1 | Install & Import Dependencies |
| 2 | Load Tokenizer |
| 3 | Configure 4-bit Quantization (QLoRA) |
| 4 | Load Frozen Base Model |
| 5 | Configure LoRA Adapters |
| 6 | Apply LoRA to Base Model |
| 7 | Load and Format HR Dataset |
| 8 | Configure Training Arguments |
| 9 | Initialize SFTTrainer |
| 10 | Run Training |
| 11 | Save LoRA Adapter Weights |
| 12 | Merge LoRA Adapters into Base Model |
| 13 | Inference Test |


---
## Step 1: Install and Import Dependencies

Install the required libraries for QLoRA fine-tuning:

- `transformers` — Hugging Face model loading and training utilities
- `peft` — Parameter-Efficient Fine-Tuning (LoRA support)
- `bitsandbytes` — 4-bit quantization engine
- `trl` — Transformer Reinforcement Learning (SFTTrainer)
- `datasets` — Dataset loading and preprocessing
- `accelerate` — Multi-GPU and mixed-precision training support


In [ ]:
# Install all required packages
!pip install transformers peft bitsandbytes trl datasets accelerate


In [ ]:
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments
)
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
from trl import SFTTrainer
from datasets import load_dataset

print("All libraries imported successfully.")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


---
## Step 2: Load Tokenizer

Load the LLaMA-2-7B-Chat tokenizer from the Hugging Face Hub.

**Key settings:**
- `pad_token = eos_token` — LLaMA-2 has no dedicated pad token; using EOS is the standard workaround for batch processing.
- `padding_side = 'right'` — Required by SFTTrainer to avoid attention mask misalignment during training.


In [ ]:
MODEL_ID = 'meta-llama/Llama-2-7b-chat-hf'

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True
)

# LLaMA-2 does not have a native pad token — use EOS as substitute
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

print(f"Tokenizer loaded: {MODEL_ID}")
print(f"Vocabulary size : {tokenizer.vocab_size:,}")
print(f"Pad token       : {tokenizer.pad_token}")
print(f"EOS token       : {tokenizer.eos_token}")


---
## Step 3: Configure 4-bit Quantization (QLoRA)

QLoRA loads the base model in **4-bit NF4 (NormalFloat4)** precision, which reduces GPU memory by ~75% compared to 16-bit loading.

| Parameter | Value | Reason |
|-----------|-------|--------|
| `load_in_4bit` | True | Activates 4-bit quantization |
| `bnb_4bit_quant_type` | `nf4` | NF4 is information-theoretically optimal for normally distributed weights |
| `bnb_4bit_compute_dtype` | `bfloat16` | BF16 offers better numerical stability than FP16 on Ampere+ GPUs |
| `bnb_4bit_use_double_quant` | True | Quantizes the quantization constants themselves — saves an additional ~0.3 bits/param |


In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit              = True,
    bnb_4bit_quant_type       = 'nf4',        # NormalFloat4 — optimal for LLM weights
    bnb_4bit_compute_dtype    = torch.bfloat16,
    bnb_4bit_use_double_quant = True,          # Double quantization for extra memory savings
)

print("BitsAndBytes 4-bit quantization config ready.")
print(f"  Quant type      : {bnb_config.bnb_4bit_quant_type}")
print(f"  Compute dtype   : {bnb_config.bnb_4bit_compute_dtype}")
print(f"  Double quant    : {bnb_config.bnb_4bit_use_double_quant}")


---
## Step 4: Load Frozen Base Model

Load LLaMA-2-7B-Chat with the 4-bit quantization config. The base model weights are **frozen** — only the LoRA adapter layers added in Step 6 will be trained.

**Key settings:**
- `use_cache = False` — Disables KV-cache during training to reduce VRAM and avoid incompatibility with gradient checkpointing.
- `pretraining_tp = 1` — Disables tensor parallelism for single-GPU training.


In [ ]:
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config = bnb_config,
    device_map          = 'auto',       # Automatically maps layers to available GPU(s)
    trust_remote_code   = True
)

# Disable KV-cache for training — incompatible with gradient checkpointing
base_model.config.use_cache = False

# Disable tensor parallelism for single-GPU setup
base_model.config.pretraining_tp = 1

print("Base model loaded and frozen.")
total_params = sum(p.numel() for p in base_model.parameters())
print(f"Total parameters: {total_params / 1e9:.2f}B")


---
## Step 5: Configure LoRA Adapters

LoRA (Low-Rank Adaptation) injects small trainable matrices into selected transformer layers. Only these adapter weights are updated during training — the base model remains frozen.

| Hyperparameter | Value | Justification |
|----------------|-------|---------------|
| `r` (rank) | 16 | Balances expressiveness vs. parameter efficiency for a narrow HR domain |
| `lora_alpha` | 32 | Standard scaling: alpha = 2 x rank |
| `lora_dropout` | 0.1 | Regularization to prevent overfitting on the small HR dataset |
| `target_modules` | q_proj, v_proj | Query and value projections yield the best quality-per-parameter trade-off |
| `bias` | none | Bias terms not updated — keeps adapter footprint minimal |


In [ ]:
lora_config = LoraConfig(
    task_type      = TaskType.CAUSAL_LM,
    r              = 16,                         # LoRA rank
    lora_alpha     = 32,                         # Scaling factor (alpha = 2 x r)
    lora_dropout   = 0.1,                        # Dropout for regularization
    target_modules = ['q_proj', 'v_proj'],       # Target attention projection layers
    bias           = 'none'                      # Do not train bias parameters
)

print("LoRA configuration ready.")
print(f"  Rank (r)        : {lora_config.r}")
print(f"  Alpha           : {lora_config.lora_alpha}")
print(f"  Dropout         : {lora_config.lora_dropout}")
print(f"  Target modules  : {lora_config.target_modules}")


---
## Step 6: Apply LoRA Adapters to Base Model

`get_peft_model()` wraps the frozen base model with the LoRA adapter layers.
After this step, only the adapter parameters (~0.01% of total) are marked as trainable.


In [ ]:
model = get_peft_model(base_model, lora_config)

# Print a summary of trainable vs. frozen parameters
model.print_trainable_parameters()

# Manual verification
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"\nTrainable params : {trainable:,}")
print(f"Total params     : {total:,}")
print(f"Trainable ratio  : {100 * trainable / total:.4f}%")


---
## Step 7: Load and Format the HR Dataset

The HR dataset consists of ~1,000 Q&A pairs structured as JSON files.
Each pair is formatted using the **LLaMA-2 chat template** before tokenization.

**LLaMA-2 Chat Template Structure:**
```
<s>[INST] <<SYS>>
{system_prompt}
<</SYS>>

{question} [/INST] {answer} </s>
```

**Expected JSON format for each record:**
```json
{
  "question": "How many casual leave days are employees entitled to?",
  "answer": "Under the Shop and Office Employees Act, employees are entitled to 7 days of casual leave per calendar year."
}
```


In [ ]:
# Load dataset from JSON files
# Replace file paths with your actual dataset locations
dataset = load_dataset('json', data_files={
    'train'     : 'hr_dataset_train.json',
    'validation': 'hr_dataset_val.json',
    'test'      : 'hr_dataset_test.json'
})

print("Dataset loaded.")
print(f"  Training samples   : {len(dataset['train'])}")
print(f"  Validation samples : {len(dataset['validation'])}")
print(f"  Test samples       : {len(dataset['test'])}")

# Preview a sample record
print("\nSample record:")
print(dataset['train'][0])


In [ ]:
SYSTEM_PROMPT = (
    'You are a professional HR Policy Assistant for a Sri Lankan company. '
    'Answer all queries accurately based on Sri Lankan labor laws and company HR policies. '
    'Maintain a formal, professional tone.'
)

def format_chat_template(example):
    """
    Formats a Q&A pair into the LLaMA-2 chat template.
    Template: <s>[INST] <<SYS>> {sys} <</SYS>> {question} [/INST] {answer} </s>
    """
    prompt = (
        f'<s>[INST] <<SYS>>\n{SYSTEM_PROMPT}\n<</SYS>>\n\n'
        f'{example["question"]} [/INST] {example["answer"]} </s>'
    )
    return {'text': prompt}

# Apply formatting across all splits
formatted = dataset.map(format_chat_template)

print("Dataset formatted with LLaMA-2 chat template.")
print("\nFormatted training example:")
print(formatted['train'][0]['text'])


---
## Step 8: Configure Training Arguments

| Argument | Value | Reason |
|----------|-------|--------|
| `num_train_epochs` | 3 | Sufficient for domain convergence on 800 training samples |
| `per_device_train_batch_size` | 4 | Fits within 16 GB VRAM with QLoRA |
| `gradient_accumulation_steps` | 4 | Effective batch size = 4 x 4 = 16 |
| `learning_rate` | 2e-4 | Standard QLoRA learning rate |
| `lr_scheduler_type` | cosine | Smooth LR decay after warmup; reduces late-training loss spikes |
| `warmup_ratio` | 0.1 | 10% of steps used for linear LR warmup |
| `optim` | paged_adamw_8bit | Memory-efficient AdamW from bitsandbytes |
| `bf16` | True | Better numerical stability on Ampere+ GPUs |
| `max_grad_norm` | 0.3 | Gradient clipping prevents exploding gradients |
| `evaluation_strategy` | epoch | Evaluate on validation set after each epoch |
| `load_best_model_at_end` | True | Automatically restores best checkpoint after training |


In [ ]:
training_args = TrainingArguments(
    output_dir                  = './hr_llama2_qlora',
    num_train_epochs            = 3,
    per_device_train_batch_size = 4,
    gradient_accumulation_steps = 4,       # Effective batch size = 16
    learning_rate               = 2e-4,
    lr_scheduler_type           = 'cosine',
    warmup_ratio                = 0.1,
    weight_decay                = 0.01,
    optim                       = 'paged_adamw_8bit',
    bf16                        = True,
    max_grad_norm               = 0.3,
    logging_steps               = 10,
    evaluation_strategy         = 'epoch',
    save_strategy               = 'epoch',
    load_best_model_at_end      = True,
    report_to                   = 'tensorboard'
)

print("Training arguments configured.")
print(f"  Epochs                     : {training_args.num_train_epochs}")
print(f"  Batch size (per device)    : {training_args.per_device_train_batch_size}")
print(f"  Gradient accumulation steps: {training_args.gradient_accumulation_steps}")
print(f"  Effective batch size       : {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"  Learning rate              : {training_args.learning_rate}")
print(f"  Optimizer                  : {training_args.optim}")


---
## Step 9: Initialize SFTTrainer

`SFTTrainer` (Supervised Fine-Tuning Trainer from TRL) is purpose-built for instruction fine-tuning of LLMs.

**Key parameters:**
- `dataset_text_field = 'text'` — Points to the formatted prompt column created in Step 7.
- `max_seq_length = 512` — Truncates/pads all sequences to 512 tokens. Covers the vast majority of HR Q&A pairs.
- `packing = False` — Processes each example independently. Prevents context leakage between Q&A pairs.


In [ ]:
trainer = SFTTrainer(
    model              = model,
    train_dataset      = formatted['train'],
    eval_dataset       = formatted['validation'],
    tokenizer          = tokenizer,
    args               = training_args,
    dataset_text_field = 'text',
    max_seq_length     = 512,
    packing            = False   # Keep each Q&A pair independent
)

print("SFTTrainer initialized and ready.")
print(f"  Training samples   : {len(formatted['train'])}")
print(f"  Validation samples : {len(formatted['validation'])}")


---
## Step 10: Run Training

Starts the QLoRA fine-tuning loop. Training updates only the LoRA adapter weights (~524K parameters).

**Expected behavior:**
- Training loss should decrease steadily from ~1.8 (epoch 1) toward ~0.9 (epoch 3).
- Validation loss should closely track training loss. A large gap indicates overfitting.
- Early stopping (patience=2) automatically selects the best checkpoint.

**Estimated training time:**
- A100 40GB GPU: ~2 hours for 3 epochs on 800 samples
- RTX 3090/4090 24GB GPU: ~3-4 hours for 3 epochs


In [ ]:
print("Starting QLoRA fine-tuning...")
print("-" * 50)

train_result = trainer.train()

print("-" * 50)
print("Training complete.")
print(f"  Total training steps : {train_result.global_step}")
print(f"  Training loss        : {train_result.training_loss:.4f}")
print(f"  Training time        : {train_result.metrics['train_runtime']:.1f} seconds")


---
## Step 11: Save LoRA Adapter Weights

Saves only the trained LoRA adapter weights (approximately 50 MB), not the full 7B base model.
This lightweight adapter can be re-applied to any fresh LLaMA-2-7B-chat base model at any time.


In [ ]:
ADAPTER_SAVE_PATH = './hr_lora_adapters'

trainer.model.save_pretrained(ADAPTER_SAVE_PATH)
tokenizer.save_pretrained(ADAPTER_SAVE_PATH)

print(f"LoRA adapter weights saved to: {ADAPTER_SAVE_PATH}")

import os
saved_files = os.listdir(ADAPTER_SAVE_PATH)
print(f"Saved files: {saved_files}")


---
## Step 12: Merge LoRA Adapters into Base Model

`merge_and_unload()` permanently fuses the LoRA adapter weights into the base model weights.
The result is a single self-contained model that **does not require the PEFT library at inference time**,
simplifying deployment significantly.

**Why merge instead of keeping adapters separate?**
- Eliminates the PEFT dependency at serving time
- Slightly faster inference (no adapter overhead)
- Easier deployment via FastAPI, vLLM, or llama.cpp
- Single model artifact to version and distribute


In [ ]:
MERGED_MODEL_PATH = './hr_llama2_merged'

print("Loading fresh base model for merging...")

# Load the base model in float16 for merging (no quantization needed at this step)
base_for_merge = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype       = torch.float16,
    trust_remote_code = True
)

# Apply and merge the saved LoRA adapters
merged_model = PeftModel.from_pretrained(base_for_merge, ADAPTER_SAVE_PATH)
merged_model = merged_model.merge_and_unload()   # Fuse adapter weights into base

# Save the fully merged model
merged_model.save_pretrained(MERGED_MODEL_PATH)
tokenizer.save_pretrained(MERGED_MODEL_PATH)

print(f"Merged model saved to: {MERGED_MODEL_PATH}")
print("The model is now ready for deployment without PEFT dependency.")


---
## Step 13: Inference Test

Run a sample HR query through the fine-tuned model to verify the output quality.

**Inference settings:**
- `temperature = 0.1` — Near-deterministic output. Low temperature is essential for factual HR policy answers.
- `max_new_tokens = 256` — Sufficient for most HR policy answers.
- `do_sample = True` — Required when temperature is set.
- `top_p = 0.9` — Nucleus sampling to prevent degenerate repetitions.


In [ ]:
from transformers import pipeline

# Load the merged model for inference
inference_model = AutoModelForCausalLM.from_pretrained(
    MERGED_MODEL_PATH,
    torch_dtype = torch.float16,
    device_map  = 'auto'
)
inference_tokenizer = AutoTokenizer.from_pretrained(MERGED_MODEL_PATH)

# Create a text generation pipeline
hr_assistant = pipeline(
    'text-generation',
    model     = inference_model,
    tokenizer = inference_tokenizer
)

print("Inference pipeline ready.")


In [ ]:
def ask_hr_assistant(question):
    """
    Query the fine-tuned HR Policy Assistant.
    Returns only the model's answer, trimmed from the full prompt.
    """
    prompt = (
        f'<s>[INST] <<SYS>>\n{SYSTEM_PROMPT}\n<</SYS>>\n\n'
        f'{question} [/INST]'
    )
    output = hr_assistant(
        prompt,
        max_new_tokens = 256,
        temperature    = 0.1,
        do_sample      = True,
        top_p          = 0.9,
        repetition_penalty = 1.1
    )
    full_text = output[0]['generated_text']
    # Extract only the model answer (after [/INST])
    answer = full_text.split('[/INST]')[-1].strip()
    return answer


# Test queries
test_questions = [
    'How many casual leave days are employees entitled to per year?',
    'What is the maternity leave entitlement under Sri Lankan law?',
    'What is the notice period required for resignation?',
    'Can unused annual leave be carried forward to the next year?'
]

print("HR Policy Assistant - Test Queries")
print("=" * 60)

for q in test_questions:
    print(f"\nQuestion: {q}")
    print(f"Answer  : {ask_hr_assistant(q)}")
    print("-" * 60)


---
## Summary

| Step | Status | Output |
|------|--------|--------|
| 1. Install & Import | Complete | Libraries loaded |
| 2. Load Tokenizer | Complete | LLaMA-2 tokenizer with EOS pad |
| 3. Quantization Config | Complete | 4-bit NF4 BitsAndBytes config |
| 4. Load Base Model | Complete | 7B params frozen, 4-bit quantized |
| 5. LoRA Config | Complete | r=16, alpha=32, q_proj + v_proj |
| 6. Apply LoRA | Complete | ~524K trainable parameters |
| 7. Load Dataset | Complete | ~1,000 HR Q&A pairs formatted |
| 8. Training Args | Complete | 3 epochs, lr=2e-4, paged_adamw_8bit |
| 9. SFTTrainer | Complete | Trainer initialized |
| 10. Train | Complete | LoRA adapters fine-tuned |
| 11. Save Adapters | Complete | Saved to ./hr_lora_adapters |
| 12. Merge Model | Complete | Saved to ./hr_llama2_merged |
| 13. Inference Test | Complete | HR queries answered |

---
